# Train the dino clip diffusion model on a Colab GPU

A flow-matching diffusion transformer that learns to generate 32-frame dino game
clips from noise. Training clips are generated on the fly by bots playing the game
inside the DataLoader workers, so there is no dataset to download.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run the cells
top to bottom. The training cell shows a live loss plot and, every few hundred
steps, clips generated from noise.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code

Re-run this cell to pull the latest code after pushing changes.

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/content/tiny-world-model")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/ayush29feb/tiny-world-model.git", str(ROOT)],
        check=True,
    )
os.chdir(ROOT)
!git log -1 --oneline

## 3. Install

Installs the repo without extras, so Colab's own CUDA build of torch is left alone.
Pillow, matplotlib and IPython are already on Colab.

In [ ]:
%pip install -q -e .

In [ ]:
# Autoreload re-imports edited code after a `git pull`, without restarting the runtime.
# Some Colab runtimes pair Python 3.13 with an IPython whose autoreload is broken
# (it imports `imp`, which Python 3.12 removed), so it is optional here.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload is on: re-run step 2 to pick up new code")
except ModuleNotFoundError:
    print(
        "autoreload isn't available on this runtime. After pulling new code, use "
        "Runtime -> Restart session, then re-run the cells from step 2."
    )

## 4. Where to save checkpoints

Colab runtimes are wiped when they disconnect. Saving to Google Drive means a new
session resumes from the last checkpoint instead of starting over. Set
`CHECKPOINT_DIR = None` to skip Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# One folder per run. A run only resumes a checkpoint made by the same model, so a
# new architecture needs a new folder (the first run, with learned positions, is in
# "dino-dit128"; resume it with DiTConfig(positions="learned")).
CHECKPOINT_DIR = "/content/drive/MyDrive/tiny-world-model/dino-dit128-rope"

## 5. Train

The loss plot and the speed plot update every `log_every` steps. Compare runs by the **val** lines, not the train line: the training loss depends on which noise levels the schedule trains on, while the validation loss uses fixed clips, fixed noise and evenly spaced noise levels. The speed plot also shows how busy the GPU is (orange) and peak GPU memory; if the GPU is well under 100% busy, a bigger batch or `compile=True` should speed things up. Every
`sample_every` steps, 8 clips are generated from the same noise with the EMA
weights, so you can watch them sharpen into the game. Those GIFs are also saved
next to the checkpoint.

Interrupting the cell is safe: re-running it resumes from the last checkpoint.

In [ ]:
from tiny_world_model.dit import DiTConfig
from tiny_world_model.flow import Schedule
from tiny_world_model.notebook import LiveMonitor
from tiny_world_model.train import TrainConfig, train

cfg = TrainConfig(
    model=DiTConfig(dim=128, depth=8, heads=4, positions="rope"),  # 3D RoPE
    steps=20_000,
    batch_size=64,
    lr=1e-3,  # peak; cosine-decays to 1e-4 by the last step
    lr_decay="cosine",
    schedule=Schedule(dist="logit_normal", shift=3.0),  # mostly high-noise levels
    compile=False,  # try True: faster steps after a few minutes of compiling
    log_every=25,
    sample_every=500,
    val_every=250,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=500,
)
ema = train(cfg, monitor=LiveMonitor(save_dir=CHECKPOINT_DIR))

## 6. Generate more clips from the trained model

In [ ]:
import base64

import torch
from IPython.display import HTML

from tiny_world_model.flow import sample
from tiny_world_model.render import clips_to_gif

clips = sample(ema, num_clips=16, steps=50, generator=torch.Generator("cuda").manual_seed(7))
gif = clips_to_gif(clips)
HTML(f"<img src='data:image/gif;base64,{base64.b64encode(gif).decode()}'>")